# 13장 - 멀티에이전트 RAG 벡터 DB 구축

원본 파일: `chap13/build_vectorstore.py`

In [8]:
import os
from glob import glob
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv

load_dotenv()

BASE_DIR = (os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd())
DATA_DIR = os.path.join(BASE_DIR, 'data')
PERSIST_DIR = os.path.join(BASE_DIR, 'chroma_store')

embedding = OpenAIEmbeddings(
    model='text-embedding-3-large',
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv('OPENAI_API_KEY'),
)

In [9]:
def read_pdf_and_split_text(pdf_path, chunk_size=1000, chunk_overlap=100):
    """주어진 PDF 파일을 읽고 텍스트를 청크 단위로 분할"""
    print(f"PDF: {pdf_path} ------------------------")

    pdf_loader = PyPDFLoader(pdf_path)
    data_from_pdf = pdf_loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    )
    splits = text_splitter.split_documents(data_from_pdf)

    print(f"Number of splits: {len(splits)}\n")
    return splits

In [10]:
def build():
    if os.path.exists(PERSIST_DIR):
        print("Loading existing Chroma store")
        vectorstore = Chroma(
            persist_directory=PERSIST_DIR,
            embedding_function=embedding,
        )
        return vectorstore

    print("Creating new Chroma store")
    vectorstore = None
    for pdf_path in glob(os.path.join(DATA_DIR, '*.pdf')):
        chunks = read_pdf_and_split_text(pdf_path)
        # 한 번에 임베딩할 청크 수를 100개씩으로 나눠서 저장 (API 요청 크기 제한 방지)
        for i in range(0, len(chunks), 100):
            if vectorstore is None:
                vectorstore = Chroma.from_documents(
                    documents=chunks[i:i + 100],
                    embedding=embedding,
                    persist_directory=PERSIST_DIR,
                )
            else:
                vectorstore.add_documents(documents=chunks[i:i + 100])

    return vectorstore

## 실행 / 테스트

In [11]:
vectorstore = build()

Loading existing Chroma store


In [12]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
chunks = retriever.invoke("서울 온실가스 저감 계획")

In [13]:
print("\n===== 검색 테스트 =====")
for chunk in chunks:
    print(chunk.metadata)
    print(chunk.page_content[:150])
    print('---')


===== 검색 테스트 =====
{'source': '/Users/seminy/Desktop/Main/Git/AI_Bootcamp/chap13/data/2040_seoul_plan.pdf', 'creationdate': '2024-12-12T18:16:11+09:00', 'pdfversion': '1.4', 'moddate': '2024-12-12T18:16:11+09:00', 'creator': 'Hwp 2020 11.0.0.5178', 'page': 63, 'page_label': '64', 'author': 'SI', 'total_pages': 205, 'producer': 'Hancom PDF 1.3.0.542'}
56제2장 미래상과 목표
6. 미래위기를 준비하는, ‘탄소중립 안전도시 구축’1) 배경전(全) 지구적인 기후변화에 대응하기 위한 대도시 차원의 대응 필요Ÿ서울시 2017년 온실가스 배출량은 46,685천 톤CO2eq로 2005년 배출량에 비해 5.6%(276만 톤CO2
---
{'author': 'SI', 'source': '/Users/seminy/Desktop/Main/Git/AI_Bootcamp/chap13/data/2040_seoul_plan.pdf', 'moddate': '2024-12-12T18:16:11+09:00', 'producer': 'Hancom PDF 1.3.0.542', 'page_label': '46', 'total_pages': 205, 'creator': 'Hwp 2020 11.0.0.5178', 'creationdate': '2024-12-12T18:16:11+09:00', 'pdfversion': '1.4', 'page': 45}
기후변화에 대응하기 위한 온실가스 적극 감축 및 녹색 인프라 확충 등을 강조하고 있다.Ÿ2040 서울도시기본계획의 부문별 전략계획과 공간계획에서는 서울의 글로벌 경쟁력 강화, 광역화에 따른 수도권 내 교통, 산업 등 네트워크 연계 강화, 기후위기에 대응하기 위한 지속가
---
{'